In [1]:
# 1
class Node:
    def __init__(self, position, parent=None):
        self.position = position
        self.parent = parent
        self.g = 0
        self.h = 0
        self.f = 0


def heuristic(current_pos, end_pos):
    # Manhattan distance
    return abs(current_pos[0] - end_pos[0]) + abs(current_pos[1] - end_pos[1])


def best_first_search(maze, start, end):
    rows = len(maze)
    cols = len(maze[0])

    start_node = Node(start)

    frontier = []
    frontier.append(start_node)

    visited = set()

    while len(frontier) > 0:
        frontier.sort(key=lambda node: node.f)

        current_node = frontier.pop(0)
        current_pos = current_node.position

        if current_pos == end:
            path = []
            while current_node:
                path.append(current_node.position)
                current_node = current_node.parent
            return path[::-1], current_pos

        visited.add(current_pos)

        # (Down, Up, Right, Left)
        for dx, dy in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            new_pos = (current_pos[0] + dx, current_pos[1] + dy)

            if 0 <= new_pos[0] < rows and 0 <= new_pos[1] < cols:
                if maze[new_pos[0]][new_pos[1]] == 0 and new_pos not in visited:
                    new_node = Node(new_pos, current_node)

                    new_node.g = current_node.g + 1
                    new_node.h = heuristic(new_pos, end)

                    new_node.f = new_node.h

                    frontier.append(new_node)
                    visited.add(new_pos)

    return None, start


maze = [
    [0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 1, 0, 1],
    [0, 0, 1, 0, 0],
    [0, 0, 0, 1, 0],
]
start = (0, 0)
end_goals = [(4, 4), (2, 1), (0, 0)]
for end in end_goals:
    print("Searching for:", end)
    path, pos = best_first_search(maze, start, end)
    if path:
        print("Path found:", path)
    start = end

Searching for: (4, 4)
Path found: [(0, 0), (1, 0), (1, 1), (1, 2), (1, 3), (2, 3), (3, 3), (3, 4), (4, 4)]
Searching for: (2, 1)
Path found: [(4, 4), (3, 4), (3, 3), (2, 3), (1, 3), (1, 2), (1, 1), (2, 1)]
Searching for: (0, 0)
Path found: [(2, 1), (1, 1), (0, 1), (0, 0)]


In [2]:
# 2
import random


def update_edge_costs(graph, change_probability=0.2):
    for node in graph:
        for neighbor in graph[node]:
            if random.random() < change_probability:
                graph[node][neighbor] = random.randint(1, 10)
    print(f"Updated Edge Costs: {graph}\n")


def reconstruct_path(came_from, current):
    path = []
    while current is not None:
        path.append(current)
        current = came_from[current]
    return path[::-1]


def dynamic_a_star(graph, start, goal, heuristic):

    frontier = []
    frontier.append((start, heuristic[start]))

    g_costs = {start: 0}
    came_from = {start: None}

    while frontier:
        frontier.sort(key=lambda x: x[1])
        current_node, current_f = frontier.pop(0)

        if current_node == goal:
            path = reconstruct_path(came_from, current_node)
            print("\nGoal found. Optimal Path:", path)
            print("Total Cost:", g_costs[goal])
            return path

        update_edge_costs(graph)

        for neighbor, cost in graph[current_node].items():
            new_g = g_costs[current_node] + cost

            if neighbor not in g_costs or new_g < g_costs[neighbor]:
                g_costs[neighbor] = new_g
                came_from[neighbor] = current_node

                f_cost = new_g + heuristic[neighbor]

                frontier.append((neighbor, f_cost))

    print("Goal not found")
    return None


graph = {
    "A": {"B": 4, "C": 3},
    "B": {"E": 12, "F": 5},
    "C": {"D": 7, "E": 10},
    "D": {"E": 2},
    "E": {"G": 5},
    "F": {"G": 16},
    "G": {},
}
heuristic = {"A": 14, "B": 12, "C": 11, "D": 6, "E": 4, "F": 11, "G": 0}

print("\nFollowing is the A* Search:")
dynamic_a_star(graph, "A", "G", heuristic)


Following is the A* Search:
Updated Edge Costs: {'A': {'B': 4, 'C': 3}, 'B': {'E': 12, 'F': 5}, 'C': {'D': 7, 'E': 10}, 'D': {'E': 3}, 'E': {'G': 7}, 'F': {'G': 16}, 'G': {}}

Updated Edge Costs: {'A': {'B': 4, 'C': 1}, 'B': {'E': 12, 'F': 5}, 'C': {'D': 7, 'E': 2}, 'D': {'E': 3}, 'E': {'G': 7}, 'F': {'G': 16}, 'G': {}}

Updated Edge Costs: {'A': {'B': 4, 'C': 1}, 'B': {'E': 12, 'F': 5}, 'C': {'D': 7, 'E': 2}, 'D': {'E': 3}, 'E': {'G': 7}, 'F': {'G': 16}, 'G': {}}


Goal found. Optimal Path: ['A', 'C', 'E', 'G']
Total Cost: 12


['A', 'C', 'E', 'G']

In [3]:
# 3
def manhattan_distance(p1, p2):
    return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])


def greedy_delivery_route(deliveries, start_location, start_time=0):

    current_location = start_location
    current_time = start_time

    route = []
    total_distance = 0

    remaining = deliveries[:]

    while remaining:
        candidates = []

        for delivery in remaining:
            location = delivery["location"]
            window_start = delivery["window"][0]
            window_end = delivery["window"][1]

            travel_time = manhattan_distance(current_location, location)

            arrival_time = current_time + travel_time

            if arrival_time <= window_end:
                # Wait if early
                effective_time = max(arrival_time, window_start)

                urgency = window_end - current_time

                priority = urgency + travel_time

                candidates.append((delivery, priority, travel_time, effective_time))

        if not candidates:
            print("No more deliveries possible within time windows.")
            break

        candidates.sort(key=lambda x: x[1])

        selected, _, travel_time, arrival_time = candidates.pop(0)

        current_time = arrival_time
        current_location = selected["location"]
        total_distance += travel_time

        route.append(selected["name"])
        remaining.remove(selected)

        print(f"Delivered to {selected['name']} at time {current_time}")

    print("\nFinal Route:", route)
    print("Total Distance Traveled:", total_distance)

    return route


deliveries = [
    {"name": "A", "location": (2, 3), "window": (2, 10)},
    {"name": "B", "location": (5, 1), "window": (0, 6)},
    {"name": "C", "location": (6, 4), "window": (5, 15)},
    {"name": "D", "location": (1, 7), "window": (3, 8)},
]

print("Delivery Route Optimization using Greedy Best-First Search\n")

greedy_delivery_route(deliveries, start_location=(0, 0), start_time=0)

Delivery Route Optimization using Greedy Best-First Search

Delivered to B at time 6
Delivered to C at time 10
No more deliveries possible within time windows.

Final Route: ['B', 'C']
Total Distance Traveled: 10


['B', 'C']